# LegalIR Task 1: Colab Single-T4 Contract Smoke Gate
## UIT Data Science Challenge 2026 — Single-GPU Topology Verification
**Pinned Git Commit:** `de507b4af75b3e8dee6bdde41e0c71a8427a5764`

### Gate Purpose:
- **Validates Single-GPU Topology (`cuda:0` / `cuda:0`)** matching production A100.
- Verifies upstream Kaggle Dual-T4 report verdict is `PASS`.
- Exercises sequential memory release between Dense and Reranker.
- Emits `colab_t4_report.json` with verdict `PASS`.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Preflight & Environment Loader
# ==============================================================================
import json
import os
import sys
import torch
from pathlib import Path

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU required for Colab T4 gate."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[+] Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
assert "T4" in gpu_name, f"Colab T4 gate requires Tesla T4 GPU (found {gpu_name})."

try:
    from google.colab import userdata
    _ud = userdata.get
except Exception:
    _ud = None
if _ud is not None:
    for _k in ["HF_TOKEN", "HF_TOKEN_WRITE", "HF_TOKEN_READ", "KAGGLE_API_TOKEN", "KAGGLE_KEY", "HF_REPO_ID"]:
        try:
            _v = _ud(_k)
        except Exception:
            _v = None
        if _v and _k not in os.environ:
            os.environ[_k] = str(_v)

for env_path in [Path("/content/.env"), Path("/content/LegalIR/.env"), Path(".env")]:
    if env_path.is_file():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip("'\""))

# Normalize HF Token (accept HF_TOKEN, HF_TOKEN_WRITE, HF_TOKEN_READ; ignore non-HF tokens like KGAT_)
hf_candidates = [os.environ.get("HF_TOKEN_WRITE"), os.environ.get("HF_TOKEN"), os.environ.get("HF_TOKEN_READ")]
hf_tok = next((t for t in hf_candidates if t and str(t).startswith("hf_")), None)
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
    try:
        from huggingface_hub import HfApi
        u_name = HfApi(token=hf_tok).whoami().get("name", "unknown")
        print(f"[+] HF_TOKEN verified (authenticated as @{u_name} for model downloads).")
    except Exception:
        print("[+] HF_TOKEN active in environment for model downloads.")

# Normalize Kaggle credentials for dataset acquisition
kg_tok = os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY")
if kg_tok and str(kg_tok).startswith("KGAT_"):
    os.environ["KAGGLE_API_TOKEN"] = kg_tok
    os.environ["KAGGLE_KEY"] = kg_tok
    kg_cfg = Path.home() / ".kaggle" / "kaggle.json"
    kg_cfg.parent.mkdir(parents=True, exist_ok=True)
    kg_user = os.environ.get("KAGGLE_USERNAME", "phucdangg")
    kg_cfg.write_text(json.dumps({"username": kg_user, "key": kg_tok}))
    kg_cfg.chmod(0o600)
    print("[+] Kaggle CLI configured from local credentials.")


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Detached HEAD Checkout
# ==============================================================================
import subprocess
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA") or "de507b4af75b3e8dee6bdde41e0c71a8427a5764"
REPO_DIR = Path("/content/LegalIR") if Path("/content").exists() else Path.cwd()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", EXPECTED_COMMIT], cwd=REPO_DIR, check=False)
    print(f"[*] Checking out exact commit: {EXPECTED_COMMIT} (detached HEAD)...")
    res = subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[*] Checkout fallback: unshallowing repository...")
        subprocess.run(["git", "fetch", "--unshallow", "origin"], cwd=REPO_DIR, check=False)
        subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[+] Working in: {REPO_DIR}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies & Dataset Setup
# Do NOT reinstall torch (Colab CUDA build). Install only missing wheels.
# ==============================================================================
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "pyvi", "pyarrow", "huggingface_hub", "bm25s", "faiss-cpu", "lightgbm", "scikit-learn"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
sys.modules["torchao"] = None
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

dataset_dir = Path("/content/kaggle_dataset") if Path("/content").exists() else REPO_DIR / "artifacts/shared/canonical/v2"
required_files = ["documents.parquet", "chunks.parquet", "queries_train.parquet", "qrels_train.parquet", "public-official.json"]
if not all((dataset_dir / _f).is_file() for _f in required_files):
    dataset_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "phucdangg/legalir-task1-clean-data", "-p", str(dataset_dir), "--unzip", "--force"], check=True)
    missing = [_f for _f in required_files if not (dataset_dir / _f).is_file()]
    assert not missing, f"Dataset incomplete after download, missing: {missing}"
print(f"[+] Dataset verified at: {dataset_dir}")


In [ ]:
# ==============================================================================
# Cell 4: Execute Colab Single-T4 Gate (scripts/gates/run_colab_t4.py)
# ==============================================================================
from pathlib import Path
from scripts.gates.run_colab_t4 import run_colab_t4_gate

output_dir = Path("/content/artifacts/task1/gates") if Path("/content").exists() else REPO_DIR / "artifacts/task1/gates"
output_dir.mkdir(parents=True, exist_ok=True)
k_cands = [
    Path("/content/kaggle_t4x2_report.json"),
    REPO_DIR / "artifacts/task1/gates/kaggle_t4x2_report.json",
    Path("/content/artifacts/task1/gates/kaggle_t4x2_report.json"),
]
k_report_path = next((p for p in k_cands if p.is_file()), k_cands[0])

report = run_colab_t4_gate(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    expected_sha=EXPECTED_COMMIT,
    kaggle_report_path=k_report_path,
    mock=False,
)
print(f"[+] Colab Single-T4 Gate execution verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Assert Gate PASS
# ==============================================================================
import json
import shutil
from pathlib import Path

report_path = output_dir / "colab_t4_report.json"
assert report_path.is_file(), f"Report missing: {report_path}"
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2))
assert report.get("verdict") == "PASS", f"Colab T4 Gate failed: {report}"
try:
    shutil.copyfile(report_path, Path("/content/colab_t4_report.json"))
except Exception:
    pass
print("\n=================================================================")
print("[+] COLAB SINGLE-T4 GATE PASSED. READY FOR PRODUCTION A100 RUN.")
print("=================================================================")
